In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("✅ Required libraries imported successfully")

✅ Required libraries imported successfully


In [2]:
MODEL_PATH = "../models/model.pkl"
VECTORIZER_PATH = "../models/vectorizer.pkl"
DATASET_PATH = "../data/complaints_clean.csv"
REPORT_PATH = "../models/final_accuracy_report.txt"

print("Model Path     :", MODEL_PATH)
print("Vectorizer Path:", VECTORIZER_PATH)
print("Dataset Path   :", DATASET_PATH)
print("Report Path    :", REPORT_PATH)

Model Path     : ../models/model.pkl
Vectorizer Path: ../models/vectorizer.pkl
Dataset Path   : ../data/complaints_clean.csv
Report Path    : ../models/final_accuracy_report.txt


In [3]:
required_files = {
    "Model": MODEL_PATH,
    "Vectorizer": VECTORIZER_PATH,
    "Dataset": DATASET_PATH
}

all_files_available = True

for file_name, file_path in required_files.items():
    if os.path.exists(file_path):
        print(f"✅ {file_name} found: {file_path}")
    else:
        print(f"❌ {file_name} not found: {file_path}")
        all_files_available = False

if all_files_available:
    print("\n✅ All required files are available")
else:
    print("\n❌ Some required files are missing. Check the paths.")

✅ Model found: ../models/model.pkl
✅ Vectorizer found: ../models/vectorizer.pkl
✅ Dataset found: ../data/complaints_clean.csv

✅ All required files are available


In [4]:
try:
    loaded_model = joblib.load(MODEL_PATH)
    loaded_vectorizer = joblib.load(VECTORIZER_PATH)

    print("✅ Model loaded successfully")
    print("✅ Vectorizer loaded successfully")

    print("\nModel Type:")
    print(type(loaded_model))

    print("\nVectorizer Type:")
    print(type(loaded_vectorizer))

except FileNotFoundError as error:
    print("❌ File not found:", error)

except Exception as error:
    print("❌ Error while loading files:", error)

✅ Model loaded successfully
✅ Vectorizer loaded successfully

Model Type:
<class 'sklearn.naive_bayes.MultinomialNB'>

Vectorizer Type:
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [5]:
try:
    df = pd.read_csv(DATASET_PATH)

    print("✅ Dataset loaded successfully")
    print("Dataset Shape:", df.shape)

    print("\nDataset Columns:")
    print(df.columns.tolist())

    print("\nFirst 5 Rows:")
    display(df.head())

except FileNotFoundError:
    print("❌ complaints_clean.csv file was not found")

except pd.errors.EmptyDataError:
    print("❌ Dataset file is empty")

except Exception as error:
    print("❌ Dataset loading error:", error)

✅ Dataset loaded successfully
Dataset Shape: (577, 9)

Dataset Columns:
['complaint_id', 'category', 'complaint_text', 'area', 'date_reported', 'priority', 'status', 'clean_text', 'tokens']

First 5 Rows:


,complaint_id,category,complaint_text,area,date_reported,priority,status,clean_text,tokens
0,1,Water Supply,"Irregular water supply timing in Melapudur, wa...",Melapudur,2026-03-11,High,In Progress,irregular water supply timing melapudur water ...,"['irregular', 'water', 'supply', 'timing', 'me..."
1,2,Road Damage,Deep crack across the road in Cantonment is wi...,Cantonment,2026-02-14,Medium,In Progress,deep crack across road cantonment widening eve...,"['deep', 'crack', 'across', 'road', 'cantonmen..."
2,3,Street Light,Broken street light glass in Anna Nagar Lake R...,Anna Nagar,2025-05-16,High,In Progress,broken street light glass anna nagar lake road...,"['broken', 'street', 'light', 'glass', 'anna',..."
3,4,Water Supply,"Irregular water supply timing in Golden Rock, ...",Golden Rock,2025-04-29,Medium,Resolved,irregular water supply timing golden rock wate...,"['irregular', 'water', 'supply', 'timing', 'go..."
4,5,Street Light,Street light near Woraiyur bus stop flickers c...,Woraiyur,2025-12-05,Low,Resolved,street light near woraiyur bus stop flickers c...,"['street', 'light', 'near', 'woraiyur', 'bus',..."


In [6]:
required_columns = ["clean_text", "category"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("❌ Missing columns:", missing_columns)
else:
    print("✅ Required columns are available")
    print("Text Column  : clean_text")
    print("Target Column: category")

✅ Required columns are available
Text Column  : clean_text
Target Column: category


In [7]:
testing_df = df[["clean_text", "category"]].copy()

print("Rows before cleaning:", len(testing_df))

testing_df = testing_df.dropna(
    subset=["clean_text", "category"]
)

testing_df["clean_text"] = (
    testing_df["clean_text"]
    .astype(str)
    .str.strip()
)

testing_df = testing_df[
    testing_df["clean_text"] != ""
]

print("Rows after cleaning :", len(testing_df))
print("Missing values      :", testing_df.isnull().sum().sum())

Rows before cleaning: 577
Rows after cleaning : 577
Missing values      : 0


In [8]:
label_encoder = LabelEncoder()

testing_df["encoded_category"] = label_encoder.fit_transform(
    testing_df["category"]
)

print("✅ Categories encoded successfully")

print("\nCategory Mapping:")

for number, category in enumerate(label_encoder.classes_):
    print(f"{number} → {category}")

✅ Categories encoded successfully

Category Mapping:
0 → Drainage
1 → Electricity
2 → Garbage
3 → Road Damage
4 → Street Light
5 → Water Supply


In [9]:
X = testing_df["clean_text"]
y = testing_df["encoded_category"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("✅ Train-test split completed")
print("Training Samples:", len(X_train_text))
print("Testing Samples :", len(X_test_text))

print("\nTraining Labels:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting Labels:")
print(pd.Series(y_test).value_counts().sort_index())

✅ Train-test split completed
Training Samples: 461
Testing Samples : 116

Training Labels:
encoded_category
0    77
1    76
2    77
3    76
4    78
5    77
Name: count, dtype: int64

Testing Labels:
encoded_category
0    19
1    19
2    19
3    19
4    20
5    20
Name: count, dtype: int64


In [10]:
try:
    X_test_tfidf = loaded_vectorizer.transform(X_test_text)

    print("✅ Test complaints transformed successfully")
    print("TF-IDF Test Shape:", X_test_tfidf.shape)

except Exception as error:
    print("❌ TF-IDF transformation error:", error)

✅ Test complaints transformed successfully
TF-IDF Test Shape: (116, 730)


In [12]:
print("Model expects features:",
      loaded_model.n_features_in_)

print("Vectorizer creates features:",
      len(loaded_vectorizer.get_feature_names_out()))

Model expects features: 263
Vectorizer creates features: 730


In [13]:
import os
import joblib

models_folder = "../models"

model_files = [
    "model.pkl",
    "best_model.pkl",
    "logistic_regression_model.pkl",
    "naive_bayes_model.pkl",
    "random_forest_model.pkl"
]

vectorizer_feature_count = len(
    loaded_vectorizer.get_feature_names_out()
)

print(
    "Current vectorizer creates:",
    vectorizer_feature_count,
    "features\n"
)

matching_models = []

for file_name in model_files:

    file_path = os.path.join(
        models_folder,
        file_name
    )

    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_name}")
        continue

    try:
        test_model = joblib.load(file_path)

        model_feature_count = getattr(
            test_model,
            "n_features_in_",
            None
        )

        print(
            f"{file_name:<35}"
            f"expects {model_feature_count} features"
        )

        if model_feature_count == vectorizer_feature_count:
            matching_models.append(file_name)
            print("   ✅ MATCHING MODEL")
        else:
            print("   ❌ Does not match")

    except Exception as error:
        print(
            f"❌ Cannot load {file_name}: {error}"
        )

print("\nMatching models:", matching_models)

Current vectorizer creates: 730 features

model.pkl                          expects 263 features
   ❌ Does not match
best_model.pkl                     expects 730 features
   ✅ MATCHING MODEL
logistic_regression_model.pkl      expects 730 features
   ✅ MATCHING MODEL
naive_bayes_model.pkl              expects 730 features
   ✅ MATCHING MODEL
random_forest_model.pkl            expects 730 features
   ✅ MATCHING MODEL

Matching models: ['best_model.pkl', 'logistic_regression_model.pkl', 'naive_bayes_model.pkl', 'random_forest_model.pkl']


In [14]:
MODEL_PATH = "../models/best_model.pkl"

loaded_model = joblib.load(MODEL_PATH)

print("✅ Matching model loaded successfully")
print("Loaded model:", MODEL_PATH)
print("Model expects:", loaded_model.n_features_in_)
print(
    "Vectorizer creates:",
    len(loaded_vectorizer.get_feature_names_out())
)

✅ Matching model loaded successfully
Loaded model: ../models/best_model.pkl
Model expects: 730
Vectorizer creates: 730


In [15]:
X_test_tfidf = loaded_vectorizer.transform(
    X_test_text
)

print("✅ Test complaints transformed successfully")
print("TF-IDF Test Shape:", X_test_tfidf.shape)

✅ Test complaints transformed successfully
TF-IDF Test Shape: (116, 730)


In [16]:
try:
    y_pred_test = loaded_model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, y_pred_test)

    print("✅ Predictions completed successfully")
    print(f"Final Model Accuracy: {accuracy * 100:.2f}%")

except ValueError as error:
    print("❌ Model and vectorizer mismatch error:")
    print(error)

except Exception as error:
    print("❌ Prediction error:", error)

✅ Predictions completed successfully
Final Model Accuracy: 100.00%


In [17]:
target_names = list(label_encoder.classes_)

report = classification_report(
    y_test,
    y_pred_test,
    target_names=target_names,
    zero_division=0
)

print("Classification Report:\n")
print(report)

Classification Report:

              precision    recall  f1-score   support

    Drainage       1.00      1.00      1.00        19
 Electricity       1.00      1.00      1.00        19
     Garbage       1.00      1.00      1.00        19
 Road Damage       1.00      1.00      1.00        19
Street Light       1.00      1.00      1.00        20
Water Supply       1.00      1.00      1.00        20

    accuracy                           1.00       116
   macro avg       1.00      1.00      1.00       116
weighted avg       1.00      1.00      1.00       116



In [18]:
confusion = confusion_matrix(
    y_test,
    y_pred_test
)

confusion_df = pd.DataFrame(
    confusion,
    index=target_names,
    columns=target_names
)

print("Confusion Matrix:")
display(confusion_df)

Confusion Matrix:


,Drainage,Electricity,Garbage,Road Damage,Street Light,Water Supply
Drainage,19,0,0,0,0,0
Electricity,0,19,0,0,0,0
Garbage,0,0,19,0,0,0
Road Damage,0,0,0,19,0,0
Street Light,0,0,0,0,20,0
Water Supply,0,0,0,0,0,20


In [19]:
new_complaints = [
    "Garbage has not been collected from our street for three days",
    "There is no water supply in Melapudur since yesterday",
    "The drainage is overflowing near the main road",
    "The street light near Anna Nagar is not working",
    "There is a deep pothole on the road in Cantonment",
    "Frequent power cuts are happening in our area"
]

try:
    new_complaints_tfidf = loaded_vectorizer.transform(
        new_complaints
    )

    new_predictions_encoded = loaded_model.predict(
        new_complaints_tfidf
    )

    new_predictions = label_encoder.inverse_transform(
        new_predictions_encoded.astype(int)
    )

    print("✅ New complaint predictions completed\n")

    for complaint, prediction in zip(
        new_complaints,
        new_predictions
    ):
        print("Complaint         :", complaint)
        print("Predicted Category:", prediction)
        print("-" * 70)

except Exception as error:
    print("❌ New complaint prediction error:", error)

✅ New complaint predictions completed

Complaint         : Garbage has not been collected from our street for three days
Predicted Category: Garbage
----------------------------------------------------------------------
Complaint         : There is no water supply in Melapudur since yesterday
Predicted Category: Water Supply
----------------------------------------------------------------------
Complaint         : The drainage is overflowing near the main road
Predicted Category: Drainage
----------------------------------------------------------------------
Complaint         : The street light near Anna Nagar is not working
Predicted Category: Street Light
----------------------------------------------------------------------
Complaint         : There is a deep pothole on the road in Cantonment
Predicted Category: Road Damage
----------------------------------------------------------------------
Complaint         : Frequent power cuts are happening in our area
Predicted Category: Elec

In [20]:
if hasattr(loaded_model, "predict_proba"):

    probabilities = loaded_model.predict_proba(
        new_complaints_tfidf
    )

    print("Prediction Confidence Scores:\n")

    for complaint, prediction, probability in zip(
        new_complaints,
        new_predictions,
        probabilities
    ):
        confidence = np.max(probability) * 100

        print("Complaint :", complaint)
        print("Category  :", prediction)
        print(f"Confidence: {confidence:.2f}%")
        print("-" * 70)

else:
    print(
        "ℹ️ This model does not support predict_proba(). "
        "Predictions are still valid."
    )

Prediction Confidence Scores:

Complaint : Garbage has not been collected from our street for three days
Category  : Garbage
Confidence: 65.00%
----------------------------------------------------------------------
Complaint : There is no water supply in Melapudur since yesterday
Category  : Water Supply
Confidence: 95.00%
----------------------------------------------------------------------
Complaint : The drainage is overflowing near the main road
Category  : Drainage
Confidence: 66.00%
----------------------------------------------------------------------
Complaint : The street light near Anna Nagar is not working
Category  : Street Light
Confidence: 100.00%
----------------------------------------------------------------------
Complaint : There is a deep pothole on the road in Cantonment
Category  : Road Damage
Confidence: 73.00%
----------------------------------------------------------------------
Complaint : Frequent power cuts are happening in our area
Category  : Electricity


In [21]:
try:
    with open(REPORT_PATH, "w", encoding="utf-8") as file:

        file.write(
            "SMART COMPLAINT MANAGEMENT SYSTEM\n"
        )
        file.write(
            "FINAL MODEL ACCURACY REPORT\n"
        )
        file.write("=" * 60 + "\n\n")

        file.write(
            f"Final Model Accuracy: "
            f"{accuracy * 100:.2f}%\n\n"
        )

        file.write("Category Mapping:\n")

        for number, category in enumerate(
            label_encoder.classes_
        ):
            file.write(
                f"{number} -> {category}\n"
            )

        file.write("\nClassification Report:\n")
        file.write(report)

        file.write("\nConfusion Matrix:\n")
        file.write(
            confusion_df.to_string()
        )

        file.write(
            "\n\nNew Complaint Predictions:\n"
        )

        for complaint, prediction in zip(
            new_complaints,
            new_predictions
        ):
            file.write(
                f"\nComplaint: {complaint}\n"
            )
            file.write(
                f"Predicted Category: {prediction}\n"
            )

    print("✅ Final accuracy report saved successfully")
    print("Saved Location:", REPORT_PATH)

except PermissionError:
    print("❌ Permission denied. Close the report file if it is open.")

except Exception as error:
    print("❌ Report saving error:", error)

✅ Final accuracy report saved successfully
Saved Location: ../models/final_accuracy_report.txt


In [22]:
print("=" * 60)
print("DAY 7 FINAL VERIFICATION")
print("=" * 60)

checks = {
    "Model file exists":
        os.path.exists(MODEL_PATH),

    "Vectorizer file exists":
        os.path.exists(VECTORIZER_PATH),

    "Dataset file exists":
        os.path.exists(DATASET_PATH),

    "Accuracy report created":
        os.path.exists(REPORT_PATH),

    "Model has predict method":
        hasattr(loaded_model, "predict"),

    "Vectorizer has transform method":
        hasattr(loaded_vectorizer, "transform"),

    "Predictions generated":
        len(new_predictions) == len(new_complaints),

    "Final accuracy calculated":
        0 <= accuracy <= 1
}

for check_name, result in checks.items():
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"{check_name:<35} {status}")

if all(checks.values()):
    print("\n🎉 DAY 7 COMPLETED SUCCESSFULLY!")
else:
    print("\n⚠️ Some Day 7 checks failed.")

DAY 7 FINAL VERIFICATION
Model file exists                   ✅ PASS
Vectorizer file exists              ✅ PASS
Dataset file exists                 ✅ PASS
Accuracy report created             ✅ PASS
Model has predict method            ✅ PASS
Vectorizer has transform method     ✅ PASS
Predictions generated               ✅ PASS
Final accuracy calculated           ✅ PASS

🎉 DAY 7 COMPLETED SUCCESSFULLY!
